# 섹션3-10. 엑셀을 활용한 머신러닝과 예측오차, Residual Plot

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 21강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 엑셀 실습

원본 엑셀 파일은 `data/raw/`에 둔다 (git에는 안 올라감, `.gitignore` 참고).

- 사용한 파일:
- 만든 차트:
- 핵심 조작(피벗, 수식 등):


### Python으로 재현 (선택)

> 강의는 본인 엑셀의 키·몸무게 데이터를 썼는데 그 원본이 없어, 같은 구조(연속형 x → 연속형 y,
> 회귀로 예측한 뒤 잔차를 보는 것)를 `viz_utils.load_sample("students")`(공부시간→점수)로
> 재현한다. 방식은 강의와 동일하게 **수식을 직접 계산**해서 예측값·잔차 컬럼을 만든다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

df = load_sample("students")[["공부시간", "점수"]]
df.head()

### 1. 산점도 + 추세선 + 수식 (엑셀의 '차트에 수식 표시')

In [ ]:
x, y = df["공부시간"].values, df["점수"].values
b, a = np.polyfit(x, y, 1)  # 엑셀 추세선과 같은 최소제곱 직선

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, s=14, alpha=0.5, color="#4C78A8")
gx = np.linspace(x.min(), x.max(), 50)
ax.plot(gx, a + b * gx, color="#E45756", lw=2)
r2 = 1 - ((y - (a + b * x)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
ax.set(xlabel="공부시간(h)", ylabel="점수",
       title=f"y = {b:.4f}x + {a:.4f}   (R² = {r2:.3f})")
plt.show()

### 2. 예측값 · 잔차 컬럼 — 엑셀에서 하던 그대로

In [ ]:
df["예측점수"] = a + b * df["공부시간"]
df["잔차"] = df["점수"] - df["예측점수"]   # 실제 - 예측
df.head()

### 3. 잔차 플롯 — X축은 원래 변수, Y축은 잔차

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["공부시간"], df["잔차"], s=14, alpha=0.5, color="#4C78A8")
ax.axhline(0, color="gray", lw=1)
ax.set(xlabel="공부시간(h)", ylabel="잔차(실제-예측)", title="잔차 플롯")
plt.show()

> 강의 결론은 "키가 커질수록 잔차가 벌어진다 → 이 모델은 키가 클 때는 못 믿는다"였다.
> 이 데이터에서도 그런지 구간별로 직접 재본다.

In [ ]:
bins = np.quantile(df["공부시간"], [0, 1/3, 2/3, 1])
for i in range(3):
    lo, hi = bins[i], bins[i + 1]
    m = (df["공부시간"] >= lo) & (df["공부시간"] <= hi if i == 2 else df["공부시간"] < hi)
    print(f"공부시간 {lo:.1f}~{hi:.1f}h (n={m.sum():3d}): 잔차 표준편차 {df.loc[m, '잔차'].std():.2f}")

> 이 데이터는 세 구간의 잔차 표준편차가 거의 비슷하다(약 8~9) — **이분산성이 없다.**
> 강의가 지적한 "x가 커질수록 잔차가 벌어지는" 패턴은 이 데이터에서는 재현되지 않는다.
> 그 패턴이 실제로 어떻게 보이는지, 일부러 그렇게 만든 예시로 대조해본다.

### 4. 대조 — 진짜 이분산적인 데이터라면

In [ ]:
# 표준편차가 x에 비례해서 커지도록 일부러 설계한 예시 (키 100~190cm처럼)
xh = rng.uniform(100, 190, 200)
noise = rng.normal(0, 0.15 * (xh - 100) + 2, 200)  # 표준편차가 x에 비례
yh = 0.9 * xh - 60 + noise

bh, ah = np.polyfit(xh, yh, 1)
resid_h = yh - (ah + bh * xh)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(xh, yh, s=14, alpha=0.5, color="#4C78A8")
gxh = np.linspace(xh.min(), xh.max(), 50)
axes[0].plot(gxh, ah + bh * gxh, color="#E45756", lw=2)
axes[0].set(xlabel="x", ylabel="y", title="산점도 — 얼핏 잘 맞아 보인다")

axes[1].scatter(xh, resid_h, s=14, alpha=0.5, color="#E45756")
axes[1].axhline(0, color="gray", lw=1)
axes[1].set(xlabel="x", ylabel="잔차", title="잔차 플롯 — 오른쪽으로 갈수록 벌어진다")
plt.tight_layout()
plt.show()

bins2 = np.quantile(xh, [0, 1/3, 2/3, 1])
for i in range(3):
    lo, hi = bins2[i], bins2[i + 1]
    m = (xh >= lo) & (xh <= hi if i == 2 else xh < hi)
    print(f"x {lo:.0f}~{hi:.0f} (n={m.sum():3d}): 잔차 표준편차 {resid_h[m].std():.2f}")

**정리.** 산점도만 보면 두 데이터 다 "선형 관계가 있다"로 끝난다. 잔차 플롯을 그려야만
구간별로 모델을 믿어도 되는지가 드러난다 — 두 번째 예시처럼 잔차가 부채꼴로 벌어지면,
강의 결론과 같은 방식으로 "x가 일정 값을 넘으면 이 회귀식은 신뢰도가 떨어진다"고 말할 수 있다.
**이 판단은 산점도가 아니라 잔차 플롯에서만 나온다.**


---

## 메모

-
